# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/a-demesa/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents the daily search performance of one content page for one client on a specific report date. This notebook uses the warehouse table that records these daily observations and supports measuring page performance over time.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features
- impressions_90d
- ctr
- avg_position
- content_age_days
- days_since_last_update

These are observable signals that are available before making a refresh decision.

### Label / Proxy
- trend_direction (used as a proxy for whether a page is declining)

This is used only as the target for learning, not as an input feature.

### Context
- month
- client_id

These provide context for grouping or filtering the data but are not part of the prediction itself.

### Excluded
- trend_pct

I deliberately exclude `trend_pct` because it directly determines the trend label. Using it as an input feature would cause data leakage and produce misleadingly high performance.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [12]:
!pip -q install datasets huggingface_hub pandas

In [13]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

login(token=HF_TOKEN)

Token loaded: True


In [14]:
from huggingface_hub import whoami
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print(whoami(token=HF_TOKEN))

{'type': 'user', 'id': '6a53945831ce3e86b70a2827', 'name': 'lbby', 'fullname': 'Angeline Alby S. de Mesa', 'email': 'angelinalbysdemesa@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1788220800, 'isPro': False, 'avatarUrl': '/avatars/d91a4fcc9de903144509d87d355c78e8.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'flyrank-read-2', 'role': 'read', 'createdAt': '2026-08-04T06:11:44.149Z'}}}


In [15]:
from datasets import load_dataset

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance"
)

dataset

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
        num_rows: 78835655
    })
})

In [16]:
train = dataset["train"]

print(train)

Dataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_rows: 78835655
})


In [17]:
print(train)
print(train.column_names)

sample = train.select(range(5))
sample.to_pandas()

Dataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_rows: 78835655
})
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chat

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


In [18]:
# Verify the unit of analysis
sample = train.select(range(5))
sample.to_pandas()[["report_date", "client_hash_id", "content_hash_id"]]

,report_date,client_hash_id,content_hash_id
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964


In [19]:
print("Total rows:", train.num_rows)

Total rows: 78835655


In [20]:
sample_df = train.select(range(10000)).to_pandas()

available = sample_df[
    (sample_df["gsc_data_available"] == True) &
    (sample_df["ga4_data_available"] == True)
]

print("Rows with both GSC and GA4 available:", len(available))

Rows with both GSC and GA4 available: 0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset provides observed website performance metrics, but it cannot explain why performance changed. It does not capture factors such as content quality, algorithm updates, competitor actions, or business decisions. Therefore, any model built from this data should be used for decision support rather than causal conclusions.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.